In [ ]:
# ============================================================
# CREDORA FINANCE — Credit Risk & Loan Approval Analytics
# Notebook 01: Exploratory Data Analysis (EDA)
# Each "# %%" block = one Jupyter cell. Paste in order.
# ============================================================

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, URL

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.edgecolor'] = '#333333'

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

In [ ]:
url = URL.create(
    drivername="mysql+mysqlconnector",
    username="root",
    password=os.environ["MYSQL_PASSWORD"],
    host="localhost",
    database="credora_risk"
)
engine = create_engine(url)

test = pd.read_sql("SELECT COUNT(*) AS n FROM customers", engine)
print("Connection OK — customers rows:", test['n'][0])

In [ ]:
customers = pd.read_sql("SELECT * FROM customers", engine)
credit_history = pd.read_sql("SELECT * FROM credit_history", engine)
loans = pd.read_sql("SELECT * FROM loan_applications", engine)
transactions = pd.read_sql("SELECT * FROM transactions", engine)

print(f"customers:      {len(customers):,} rows")
print(f"credit_history: {len(credit_history):,} rows")
print(f"loans:          {len(loans):,} rows")
print(f"transactions:   {len(transactions):,} rows")

# Expected: 5,000 / 5,000 / 7,500 / 40,079

In [ ]:
for name, df in [('customers', customers), ('credit_history', credit_history),
                 ('loans', loans), ('transactions', transactions)]:
    print(f"\n=== {name} ===")
    print(f"Duplicates: {df.duplicated().sum()}")
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print("Missing values:")
        print(missing[missing > 0])
    else:
        print("No missing values")

# Expected missing: loans.decision_date = 173 (Under Review),
# loans.loan_default = 4097 (non-disbursed). Everything else should be clean.

In [ ]:
disbursed = loans[loans['disbursed_flag'] == 1]

print(f"Disbursed loans: {len(disbursed):,}")
print(f"Observed default rate: {disbursed['loan_default'].mean()*100:.1f}%")
print(f"\nApproval status distribution:")
print(loans['approval_status'].value_counts())
decided = loans[loans['approval_status'].isin(['Approved', 'Rejected'])]
print(f"\nApproval rate (decided apps): "
      f"{(decided['approval_status'] == 'Approved').mean()*100:.1f}%")

# Expected: ~3,403 disbursed, ~23.4% default, ~46.4% approval

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(credit_history['credit_score'], bins=40,
             color='steelblue', edgecolor='white')
axes[0].set_title('Credit Score Distribution')
axes[0].set_xlabel('Credit Score')
axes[0].set_ylabel('Customers')

merged = disbursed.merge(
    credit_history[['customer_id', 'credit_score']], on='customer_id')
merged['band'] = pd.cut(
    merged['credit_score'],
    bins=[300, 580, 670, 740, 800, 900],
    labels=['<580', '580-669', '670-739', '740-799', '800+'])
merged.groupby('band', observed=True)['loan_default'].mean().mul(100).plot(
    kind='bar', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Default Rate % by Credit Score Band')
axes[1].set_ylabel('Default Rate %')
axes[1].tick_params(axis='x', rotation=0)

loans['loan_type'].value_counts().plot(
    kind='bar', ax=axes[2], color='seagreen', edgecolor='white')
axes[2].set_title('Applications by Loan Type')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# KEY CHECK: middle chart — default rate should RISE as credit score falls.

In [ ]:
loans['app_month'] = pd.to_datetime(loans['application_date']).dt.to_period('M')
monthly = loans.groupby('app_month').agg(
    applications=('application_id', 'count'),
    approved=('approval_status', lambda s: (s == 'Approved').sum())
)
monthly['approval_rate'] = monthly['approved'] / monthly['applications'] * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
monthly['applications'].plot(ax=axes[0], marker='o', color='steelblue')
axes[0].set_title('Monthly Application Volume')
axes[0].set_ylabel('Applications')

monthly['approval_rate'].plot(ax=axes[1], marker='o', color='darkorange')
axes[1].set_title('Monthly Approval Rate %')
axes[1].set_ylabel('Approval Rate %')

plt.tight_layout()
plt.show()

In [ ]:
(disbursed.groupby('loan_type')['loan_default'].mean().mul(100)
 .sort_values(ascending=False)
 .plot(kind='bar', color='coral', edgecolor='white',
       title='Default Rate % by Loan Type (disbursed only)'))
plt.ylabel('Default Rate %')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(customers['annual_income'], bins=40,
             color='seagreen', edgecolor='white')
axes[0].set_title('Annual Income Distribution')
axes[0].set_xlabel('Annual Income (INR)')

axes[1].hist(customers['age'], bins=30, color='slateblue', edgecolor='white')
axes[1].set_title('Age Distribution')
axes[1].set_xlabel('Age')

customers['employment_type'].value_counts().plot(
    kind='bar', ax=axes[2], color='darkorange', edgecolor='white')
axes[2].set_title('Employment Type')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
txn_summary = transactions.groupby('category')['amount'].agg(
    ['count', 'sum', 'mean']).sort_values('sum', ascending=False)
print("Transaction volume by category:")
print(txn_summary.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
txn_summary['sum'].plot(kind='bar', ax=axes[0], color='steelblue',
                        edgecolor='white')
axes[0].set_title('Total Amount by Category')
axes[0].tick_params(axis='x', rotation=45)

transactions['channel'].value_counts().plot(
    kind='bar', ax=axes[1], color='seagreen', edgecolor='white')
axes[1].set_title('Transactions by Channel')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
corr_df = disbursed.merge(credit_history, on='customer_id')
num_cols = ['credit_score', 'credit_utilization_ratio',
            'num_open_accounts', 'num_credit_inquiries_6m',
            'num_late_payments_30d', 'num_late_payments_90d',
            'num_defaults_prior', 'bankruptcies',
            'total_outstanding_debt', 'debt_to_income_ratio',
            'interest_rate', 'requested_amount', 'loan_term_months',
            'loan_default']

corr = corr_df[num_cols].corr()

plt.figure(figsize=(11, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix (disbursed loans)')
plt.tight_layout()
plt.show()

print("\nTop correlations with loan_default:")
print(corr['loan_default'].drop('loan_default')
      .sort_values(key=abs, ascending=False).round(3))

# KEY CHECK: credit_score correlation should be NEGATIVE,
# utilization / prior defaults / DTI should be POSITIVE.

In [ ]:
model_base = (loans
              .merge(credit_history, on='customer_id', how='left')
              .merge(customers.drop(columns=['email', 'phone']),
                     on='customer_id', how='left'))

model_base.to_csv('credora_model_base.csv', index=False)
print(f"Saved credora_model_base.csv: "
      f"{model_base.shape[0]:,} rows × {model_base.shape[1]} columns")

In [ ]:
print("""
EDA COMPLETE — key findings to write in your notebook markdown:
1. Portfolio: 5,000 customers, 7,500 applications, 3,403 disbursed.
2. Observed default rate: 23.4% — a stressed book; risk analytics justified.
3. Approval rate: 46.4% of decided applications.
4. Default rate rises sharply as credit score falls (see Cell 6).
5. Strongest default drivers from correlation: see Cell 11 output.
6. Data quality: 0 duplicates, missing values only where legitimate
   (decision_date for pending, loan_default for non-disbursed).
NEXT: Notebook 02 — feature engineering (aggregate transactions to
customer grain, encode categoricals, handle leakage).
""")